# Смотрим на данные: реклама велосипедов

Выгрузка из Директа по магазину велосипедов (Centra Market, Иркутск и область).
Одна таблица: и срезы аудитории, и метрики показов.



In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import pickle


### Работа с вещественными признаками

In [2]:
DATA_PATH = "../dataset/bycicles.csv"
data = pd.read_csv(DATA_PATH)
print(data.shape)
data.head(5)


(9923, 28)


,Дата,Группа,№ Группы,№ Объявления,Тип площадки,Формат,Размер изображения,Тип устройства,Пол,Категория таргетинга,...,Клики,CTR (%),wCTR (%),Расход (руб.),Конверсия (%)/Товар добавлен в корзину,Конверсия (%)/Заказ оформлен,Конверсия (%)/Лид с Centra Market//Динамика,Конверсии/Товар добавлен в корзину,Конверсии/Заказ оформлен,Конверсии/Лид с Centra Market//Динамика
0,15.06.2023,Велосипед Stinger,4296804875,M-9591380676,поиск,текстовый,без изображения,мобильные,не определен,Целевые запросы,...,0,0.0,0.0,0.0,-,-,-,-,-,-
1,15.06.2023,Велосипед Stinger,4296804875,M-9591380678,поиск,графический,с изображением,десктоп,мужской,Запросы с упоминанием конкурентов,...,0,0.0,0.0,0.0,-,-,-,-,-,-
2,15.06.2023,Велосипед Женский,4296804877,M-9591380694,поиск,графический,с изображением,десктоп,женский,Целевые запросы,...,0,0.0,0.0,0.0,-,-,-,-,-,-
3,15.06.2023,Велосипед Женский,4296804877,M-9591380696,поиск,графический,с изображением,десктоп,женский,Целевые запросы,...,0,0.0,0.0,0.0,-,-,-,-,-,-
4,15.06.2023,Велосипед Складной,4296804880,M-9591380710,поиск,текстовый,без изображения,мобильные,мужской,Целевые запросы,...,0,0.0,0.0,0.0,-,-,-,-,-,-


#### Чистим данные

In [3]:
# Берём кусок таблицы, чтобы pairplot не считался вечность
data = pd.read_csv(DATA_PATH)[:100]
data.head(10)


,Дата,Группа,№ Группы,№ Объявления,Тип площадки,Формат,Размер изображения,Тип устройства,Пол,Категория таргетинга,...,Клики,CTR (%),wCTR (%),Расход (руб.),Конверсия (%)/Товар добавлен в корзину,Конверсия (%)/Заказ оформлен,Конверсия (%)/Лид с Centra Market//Динамика,Конверсии/Товар добавлен в корзину,Конверсии/Заказ оформлен,Конверсии/Лид с Centra Market//Динамика
0,15.06.2023,Велосипед Stinger,4296804875,M-9591380676,поиск,текстовый,без изображения,мобильные,не определен,Целевые запросы,...,0,0.0,0.00,0.00,-,-,-,-,-,-
1,15.06.2023,Велосипед Stinger,4296804875,M-9591380678,поиск,графический,с изображением,десктоп,мужской,Запросы с упоминанием конкурентов,...,0,0.0,0.00,0.00,-,-,-,-,-,-
2,15.06.2023,Велосипед Женский,4296804877,M-9591380694,поиск,графический,с изображением,десктоп,женский,Целевые запросы,...,0,0.0,0.00,0.00,-,-,-,-,-,-
3,15.06.2023,Велосипед Женский,4296804877,M-9591380696,поиск,графический,с изображением,десктоп,женский,Целевые запросы,...,0,0.0,0.00,0.00,-,-,-,-,-,-
4,15.06.2023,Велосипед Складной,4296804880,M-9591380710,поиск,текстовый,без изображения,мобильные,мужской,Целевые запросы,...,0,0.0,0.00,0.00,-,-,-,-,-,-
5,15.06.2023,Велосипед Детский,4296804882,M-9591380722,поиск,текстовый,без изображения,мобильные,женский,Целевые запросы,...,1,100.0,150.17,26.02,-,-,-,-,-,-
6,15.06.2023,Велосипед Детский,4296804882,M-9591380722,поиск,текстовый,без изображения,мобильные,женский,Целевые запросы,...,0,0.0,0.00,0.00,-,-,-,-,-,-
7,15.06.2023,Велосипед Детский,4296804882,M-9591380722,поиск,текстовый,без изображения,мобильные,женский,Целевые запросы,...,1,100.0,146.31,18.21,-,-,-,-,-,-
8,15.06.2023,Велосипед Детский,4296804882,M-9591380724,поиск,графический,с изображением,десктоп,мужской,Целевые запросы,...,1,100.0,97.10,41.47,-,-,-,-,-,-
9,15.06.2023,Велосипед Детский,4296804882,M-9591380724,поиск,графический,с изображением,десктоп,женский,Целевые запросы,...,1,100.0,138.12,15.49,-,-,-,-,-,-


In [4]:
data.info()


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 28 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   Дата                                         100 non-null    str    
 1   Группа                                       100 non-null    str    
 2   № Группы                                     100 non-null    int64  
 3   № Объявления                                 100 non-null    str    
 4   Тип площадки                                 100 non-null    str    
 5   Формат                                       100 non-null    str    
 6   Размер изображения                           100 non-null    str    
 7   Тип устройства                               100 non-null    str    
 8   Пол                                          100 non-null    str    
 9   Категория таргетинга                         100 non-null    str    
 10  Упоминание бре

In [5]:
# Прочерки в конверсиях — это не число. Пустые и константные колонки тоже не признаки.
dash_cols = [c for c in data.columns if data[c].dtype == "object" and (data[c] == "-").any()]
for col in dash_cols:
    data[col] = pd.to_numeric(data[col].mask(data[col] == "-"), errors="coerce")

empty_or_const = [c for c in data.columns if data[c].nunique(dropna=True) <= 1]
print("убрали:", empty_or_const)
data = data.drop(columns=empty_or_const)
data.head(3)


убрали: ['Дата', 'Тип площадки', 'Упоминание брендов', 'Ссылка', 'Конверсия (%)/Товар добавлен в корзину', 'Конверсия (%)/Заказ оформлен', 'Конверсия (%)/Лид с Centra Market//Динамика', 'Конверсии/Товар добавлен в корзину', 'Конверсии/Заказ оформлен', 'Конверсии/Лид с Centra Market//Динамика']


,Группа,№ Группы,№ Объявления,Формат,Размер изображения,Тип устройства,Пол,Категория таргетинга,Уровень платежеспособности,Возраст,Заголовок,Текст,Показы,Взвешенные показы,Клики,CTR (%),wCTR (%),Расход (руб.)
0,Велосипед Stinger,4296804875,M-9591380676,текстовый,без изображения,мобильные,не определен,Целевые запросы,Остальные,не определен,Магазин велосипедов Stinger,Велосипеды Stinger в Иркутске и обл. От 11 450...,1,1.00,0,0.0,0.0,0.0
1,Велосипед Stinger,4296804875,M-9591380678,графический,с изображением,десктоп,мужской,Запросы с упоминанием конкурентов,Остальные,35-44,{PHRASEКупить велосипед Stinger},Велосипеды Stinger в Иркутске и обл. От 11 450...,1,0.81,0,0.0,0.0,0.0
2,Велосипед Женский,4296804877,M-9591380694,графический,с изображением,десктоп,женский,Целевые запросы,Остальные,25-34,{PHRASEКупить женский велосипед},Женские велосипеды в Иркутске. Доставка. От 92...,1,0.09,0,0.0,0.0,0.0


In [6]:
# Оставляем только вещественные признаки. Номер группы — идентификатор, не метрика.
columns_with_numeric = data.select_dtypes(include=["float64", "int64"]).columns
id_like = [c for c in columns_with_numeric if c.startswith("№")]
columns_with_values = data[columns_with_numeric].drop(columns=id_like, errors="ignore")
columns_with_values = columns_with_values.dropna(how="all", axis=1).columns

cleaned_data = data[columns_with_values]
cleaned_data


,Показы,Взвешенные показы,Клики,CTR (%),wCTR (%),Расход (руб.)
0,1,1.00,0,0.0,0.0,0.0
1,1,0.81,0,0.0,0.0,0.0
2,1,0.09,0,0.0,0.0,0.0
3,1,0.09,0,0.0,0.0,0.0
4,1,0.70,0,0.0,0.0,0.0
...,...,...,...,...,...,...
95,1,0.09,0,0.0,0.0,0.0
96,1,0.81,0,0.0,0.0,0.0
97,2,1.89,0,0.0,0.0,0.0
98,1,0.94,0,0.0,0.0,0.0


#### Строим всякие распределения

In [7]:
import seaborn as sns
import matplotlib.pyplot as plt


In [8]:
sns.pairplot(cleaned_data)
plt.show()


#### Выбираем только самые важные признаки

Вопросы на подумать: - Как выбрать признаки, которые можно убрать из колонок?

##### Убираем по threshold взаимной корреляции

In [9]:
correlation_matrix = cleaned_data.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True)

plt.title("Correlation Matrix", fontsize=16)
plt.show()


In [10]:
threshold = 0.85
correlation_matrix = cleaned_data.corr()

high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if abs(correlation_matrix.iloc[i, j]) > threshold:
            colname_i = correlation_matrix.columns[i]
            colname_j = correlation_matrix.columns[j]
            high_corr_pairs.append((colname_i, colname_j, correlation_matrix.iloc[i, j]))

high_corr_pairs


[('Взвешенные показы', 'Показы', np.float64(0.9054341757175474)),
 ('CTR (%)', 'Клики', np.float64(0.9062435134903571)),
 ('wCTR (%)', 'Клики', np.float64(0.8577640712375761)),
 ('wCTR (%)', 'CTR (%)', np.float64(0.9712560847304228)),
 ('Расход (руб.)', 'Клики', np.float64(0.9147628820478698))]

In [11]:
# На первых 100 строках показы почти совпадают со взвешенными,
# клики — с CTR, wCTR и расходом. Оставляем одну сторону каждой пары.
to_remove = ["Взвешенные показы", "wCTR (%)", "Расход (руб.)"]


In [12]:
filtered_data = cleaned_data.drop(columns=to_remove)
sns.pairplot(filtered_data)
plt.show()


### Другой вариант отбора признаков

VIF — это классический статистический метод для выявления **мультиколлинеарности**. Давай разберёмся по шагам.

---

#### Теория: что такое VIF

* Для каждого признака $X_i$ мы строим регрессию на все остальные признаки:

$$
R_i^2 = R^2(X_i \sim X_{-i})
$$

* То есть смотрим, насколько хорошо $X_i$ объясняется комбинацией других признаков.

* Если признак сильно «дублируется» другими, то $R_i^2$ будет близок к 1.

* **Формула VIF**:

$$
VIF_i = \frac{1}{1 - R_i^2}
$$

* Интерпретация:

  * $VIF \approx 1$ → признак независим.
  * $VIF > 5$ → подозрение на сильную корреляцию.
  * $VIF > 10$ → мультиколлинеарность, признак стоит убрать.

---

#### Почему это работает

* Если признак «уникальный» → другие не могут его предсказать → $R^2$ низкий → $VIF \approx 1$.
* Если признак «лишний» (почти линейная комбинация других) → $R^2 \to 1$ → знаменатель $(1-R^2) \to 0$ → $VIF \to \infty$.
* Таким образом, VIF прямо показывает «насколько этот признак избыточен».


In [13]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant


def calculate_vif(df):
    """Считает Variance Inflation Factor (VIF) для числовых признаков.

    Args:
        df (pd.DataFrame): датафрейм с числовыми фичами

    Returns:
        pd.DataFrame: таблица с VIF для каждого признака
    """
    # Добавляем константу (обязательно для корректного расчёта)
    X = add_constant(df)

    vif_data = []
    for i in range(1, X.shape[1]):  # начинаем с 1, чтобы пропустить константу
        vif = variance_inflation_factor(X.values, i)
        vif_data.append((df.columns[i - 1], vif))

    return pd.DataFrame(vif_data, columns=["Feature", "VIF"]).sort_values(by="VIF", ascending=False)


In [14]:
# Считаем VIF
vif_table = calculate_vif(cleaned_data)
print(vif_table.round(2))


             Feature    VIF
3            CTR (%)  34.81
4           wCTR (%)  31.56
2              Клики  24.58
5      Расход (руб.)  12.24
1  Взвешенные показы   6.33
0             Показы   5.81


In [15]:
# VIF высокий у кликов, CTR и wCTR: они почти линейно выражают друг друга.
to_remove = ["Клики", "CTR (%)", "wCTR (%)"]


In [16]:
filtered_data = cleaned_data.drop(columns=to_remove)
sns.pairplot(filtered_data)
plt.show()


##### Вопросы на подумать: - Какие зависимости на ваш взгляд кажутся информативными?

### Работа с категориальными данными

In [17]:
data = pd.read_csv(DATA_PATH)
data.head(10)


,Дата,Группа,№ Группы,№ Объявления,Тип площадки,Формат,Размер изображения,Тип устройства,Пол,Категория таргетинга,...,Клики,CTR (%),wCTR (%),Расход (руб.),Конверсия (%)/Товар добавлен в корзину,Конверсия (%)/Заказ оформлен,Конверсия (%)/Лид с Centra Market//Динамика,Конверсии/Товар добавлен в корзину,Конверсии/Заказ оформлен,Конверсии/Лид с Centra Market//Динамика
0,15.06.2023,Велосипед Stinger,4296804875,M-9591380676,поиск,текстовый,без изображения,мобильные,не определен,Целевые запросы,...,0,0.0,0.00,0.00,-,-,-,-,-,-
1,15.06.2023,Велосипед Stinger,4296804875,M-9591380678,поиск,графический,с изображением,десктоп,мужской,Запросы с упоминанием конкурентов,...,0,0.0,0.00,0.00,-,-,-,-,-,-
2,15.06.2023,Велосипед Женский,4296804877,M-9591380694,поиск,графический,с изображением,десктоп,женский,Целевые запросы,...,0,0.0,0.00,0.00,-,-,-,-,-,-
3,15.06.2023,Велосипед Женский,4296804877,M-9591380696,поиск,графический,с изображением,десктоп,женский,Целевые запросы,...,0,0.0,0.00,0.00,-,-,-,-,-,-
4,15.06.2023,Велосипед Складной,4296804880,M-9591380710,поиск,текстовый,без изображения,мобильные,мужской,Целевые запросы,...,0,0.0,0.00,0.00,-,-,-,-,-,-
5,15.06.2023,Велосипед Детский,4296804882,M-9591380722,поиск,текстовый,без изображения,мобильные,женский,Целевые запросы,...,1,100.0,150.17,26.02,-,-,-,-,-,-
6,15.06.2023,Велосипед Детский,4296804882,M-9591380722,поиск,текстовый,без изображения,мобильные,женский,Целевые запросы,...,0,0.0,0.00,0.00,-,-,-,-,-,-
7,15.06.2023,Велосипед Детский,4296804882,M-9591380722,поиск,текстовый,без изображения,мобильные,женский,Целевые запросы,...,1,100.0,146.31,18.21,-,-,-,-,-,-
8,15.06.2023,Велосипед Детский,4296804882,M-9591380724,поиск,графический,с изображением,десктоп,мужской,Целевые запросы,...,1,100.0,97.10,41.47,-,-,-,-,-,-
9,15.06.2023,Велосипед Детский,4296804882,M-9591380724,поиск,графический,с изображением,десктоп,женский,Целевые запросы,...,1,100.0,138.12,15.49,-,-,-,-,-,-


In [18]:
# Таблица небольшая — для категорий берём все строки.
# В первых 1000 почти нет упоминания брендов и конверсий.
data = pd.read_csv(DATA_PATH)
data.head(5)


,Дата,Группа,№ Группы,№ Объявления,Тип площадки,Формат,Размер изображения,Тип устройства,Пол,Категория таргетинга,...,Клики,CTR (%),wCTR (%),Расход (руб.),Конверсия (%)/Товар добавлен в корзину,Конверсия (%)/Заказ оформлен,Конверсия (%)/Лид с Centra Market//Динамика,Конверсии/Товар добавлен в корзину,Конверсии/Заказ оформлен,Конверсии/Лид с Centra Market//Динамика
0,15.06.2023,Велосипед Stinger,4296804875,M-9591380676,поиск,текстовый,без изображения,мобильные,не определен,Целевые запросы,...,0,0.0,0.0,0.0,-,-,-,-,-,-
1,15.06.2023,Велосипед Stinger,4296804875,M-9591380678,поиск,графический,с изображением,десктоп,мужской,Запросы с упоминанием конкурентов,...,0,0.0,0.0,0.0,-,-,-,-,-,-
2,15.06.2023,Велосипед Женский,4296804877,M-9591380694,поиск,графический,с изображением,десктоп,женский,Целевые запросы,...,0,0.0,0.0,0.0,-,-,-,-,-,-
3,15.06.2023,Велосипед Женский,4296804877,M-9591380696,поиск,графический,с изображением,десктоп,женский,Целевые запросы,...,0,0.0,0.0,0.0,-,-,-,-,-,-
4,15.06.2023,Велосипед Складной,4296804880,M-9591380710,поиск,текстовый,без изображения,мобильные,мужской,Целевые запросы,...,0,0.0,0.0,0.0,-,-,-,-,-,-


In [19]:
# Убираем колонки, в которых все данные одинаковые или пустые
columns_to_drop = [c for c in data.columns if data[c].nunique(dropna=True) <= 1]
data = data.drop(columns=columns_to_drop)

# Посмотрим, какие колонки были удалены
columns_to_drop


['Тип площадки',
 'Ссылка',
 'Конверсия (%)/Заказ оформлен',
 'Конверсии/Заказ оформлен']

In [20]:
print(data.keys())
data.head(2)


Index(['Дата', 'Группа', '№ Группы', '№ Объявления', 'Формат',
       'Размер изображения', 'Тип устройства', 'Пол', 'Категория таргетинга',
       'Упоминание брендов', 'Уровень платежеспособности', 'Возраст',
       'Заголовок', 'Текст', 'Показы', 'Взвешенные показы', 'Клики', 'CTR (%)',
       'wCTR (%)', 'Расход (руб.)', 'Конверсия (%)/Товар добавлен в корзину',
       'Конверсия (%)/Лид с Centra Market//Динамика',
       'Конверсии/Товар добавлен в корзину',
       'Конверсии/Лид с Centra Market//Динамика'],
      dtype='str')


,Дата,Группа,№ Группы,№ Объявления,Формат,Размер изображения,Тип устройства,Пол,Категория таргетинга,Упоминание брендов,...,Показы,Взвешенные показы,Клики,CTR (%),wCTR (%),Расход (руб.),Конверсия (%)/Товар добавлен в корзину,Конверсия (%)/Лид с Centra Market//Динамика,Конверсии/Товар добавлен в корзину,Конверсии/Лид с Centra Market//Динамика
0,15.06.2023,Велосипед Stinger,4296804875,M-9591380676,текстовый,без изображения,мобильные,не определен,Целевые запросы,не определено,...,1,1.00,0,0.0,0.0,0.0,-,-,-,-
1,15.06.2023,Велосипед Stinger,4296804875,M-9591380678,графический,с изображением,десктоп,мужской,Запросы с упоминанием конкурентов,не определено,...,1,0.81,0,0.0,0.0,0.0,-,-,-,-


### Какие вопросы можно задать к данным?

Соотношение мужчин и женщин разного возраста?

Как связаны формат объявления, картинка и тип устройства?

Уровень платежеспособности и тип устройства.

Категория таргетинга и пол.

Группа (что продают) в зависимости от возраста.


In [21]:
import matplotlib.gridspec as gridspec

top_groups = data["Группа"].value_counts().head(6).index
plot_data = data[data["Группа"].isin(top_groups)]

plt.figure(figsize=(15, 15))
gs = gridspec.GridSpec(3, 2, height_ratios=[1, 1, 1])

ax1 = plt.subplot(gs[0, 0])
sns.countplot(data=data, x="Возраст", hue="Пол", palette="Blues", ax=ax1)
ax1.set_title("Соотношение мужчин и женщин разного возраста")
ax1.set_ylabel("Количество")
ax1.set_xlabel("Возраст")
ax1.tick_params(axis="x", rotation=30)

ax2 = plt.subplot(gs[0, 1])
sns.countplot(data=plot_data, x="Возраст", hue="Группа", palette="Greens", ax=ax2)
ax2.set_title("Группа в зависимости от возраста (топ-6 групп)")
ax2.set_ylabel("Количество")
ax2.set_xlabel("Возраст")
ax2.tick_params(axis="x", rotation=30)

ax3 = plt.subplot(gs[1, 0])
sns.countplot(data=data, x="Тип устройства", hue="Формат", palette="Oranges", ax=ax3)
ax3.set_title("Формат объявления и тип устройства")
ax3.set_ylabel("Количество")
ax3.set_xlabel("Тип устройства")

ax4 = plt.subplot(gs[1, 1])
sns.countplot(data=data, x="Тип устройства", hue="Уровень платежеспособности", palette="YlOrBr", ax=ax4)
ax4.set_title("Платежеспособность и тип устройства")
ax4.set_ylabel("Количество")
ax4.set_xlabel("Тип устройства")

ax5 = plt.subplot(gs[2, 0])
sns.countplot(data=data, x="Категория таргетинга", hue="Пол", palette="YlOrRd", ax=ax5)
ax5.set_title("Таргетинг и пол")
ax5.set_ylabel("Количество")
ax5.set_xlabel("Категория таргетинга")
ax5.tick_params(axis="x", rotation=40)

ax6 = plt.subplot(gs[2, 1])
sns.countplot(data=data, x="Формат", hue="Размер изображения", palette="Purples", ax=ax6)
ax6.set_title("Формат и картинка — одна и та же нарезка")
ax6.set_ylabel("Количество")
ax6.set_xlabel("Формат")

plt.tight_layout()
plt.show()


### Работа с категориальными данными

#### Как обработать данные?

#### Label encoding

In [22]:
# Кодируем категории, числа и дату не трогаем.
# Текст объявления оставляем как есть — это не уровень шкалы.
raw_data = data.copy()

skip = {"Дата", "Заголовок", "Текст"} | set(raw_data.select_dtypes(include=["number"]).columns)
cat_columns = [c for c in raw_data.columns if c not in skip]

encoding_dict = {}
encoded = raw_data.copy()
for column in cat_columns:
    le = LabelEncoder()
    encoded[column] = le.fit_transform(encoded[column].astype(str))
    encoding_dict[column] = dict(zip(le.classes_, le.transform(le.classes_)))

pickle_file_path = "../dataset/encoding_dict.pkl"
with open(pickle_file_path, "wb") as f:
    pickle.dump(encoding_dict, f)

encoded.to_csv("../dataset/label_ecoded_1000.csv")
encoded.head()


,Дата,Группа,№ Группы,№ Объявления,Формат,Размер изображения,Тип устройства,Пол,Категория таргетинга,Упоминание брендов,...,Показы,Взвешенные показы,Клики,CTR (%),wCTR (%),Расход (руб.),Конверсия (%)/Товар добавлен в корзину,Конверсия (%)/Лид с Centra Market//Динамика,Конверсии/Товар добавлен в корзину,Конверсии/Лид с Centra Market//Динамика
0,15.06.2023,7,4296804875,0,1,0,1,2,4,3,...,1,1.00,0,0.0,0.0,0.0,0,0,0,0
1,15.06.2023,7,4296804875,1,0,1,0,1,1,3,...,1,0.81,0,0.0,0.0,0.0,0,0,0,0
2,15.06.2023,14,4296804877,5,0,1,0,0,4,3,...,1,0.09,0,0.0,0.0,0.0,0,0,0,0
3,15.06.2023,14,4296804877,6,0,1,0,0,4,3,...,1,0.09,0,0.0,0.0,0.0,0,0,0,0
4,15.06.2023,23,4296804880,13,1,0,1,1,4,3,...,1,0.70,0,0.0,0.0,0.0,0,0,0,0


#### OneHot encoding

In [23]:
date_column = raw_data.columns[0]
exclude_columns = ["Заголовок", "Текст"]
columns_to_encode = [
    col for col in raw_data.columns
    if col not in exclude_columns + [date_column]
    and raw_data[col].dtype == "object"
]
df_encoded = pd.get_dummies(raw_data, columns=columns_to_encode)
df_encoded.head(3)


,Дата,Группа,№ Группы,№ Объявления,Формат,Размер изображения,Тип устройства,Пол,Категория таргетинга,Упоминание брендов,...,Показы,Взвешенные показы,Клики,CTR (%),wCTR (%),Расход (руб.),Конверсия (%)/Товар добавлен в корзину,Конверсия (%)/Лид с Centra Market//Динамика,Конверсии/Товар добавлен в корзину,Конверсии/Лид с Centra Market//Динамика
0,15.06.2023,Велосипед Stinger,4296804875,M-9591380676,текстовый,без изображения,мобильные,не определен,Целевые запросы,не определено,...,1,1.00,0,0.0,0.0,0.0,-,-,-,-
1,15.06.2023,Велосипед Stinger,4296804875,M-9591380678,графический,с изображением,десктоп,мужской,Запросы с упоминанием конкурентов,не определено,...,1,0.81,0,0.0,0.0,0.0,-,-,-,-
2,15.06.2023,Велосипед Женский,4296804877,M-9591380694,графический,с изображением,десктоп,женский,Целевые запросы,не определено,...,1,0.09,0,0.0,0.0,0.0,-,-,-,-


In [24]:
column_mapping = {col: f"col_{i + 1}" for i, col in enumerate(df_encoded.columns)}
df_renamed = df_encoded.rename(columns=column_mapping)
column_mapping


{'Дата': 'col_1',
 'Группа': 'col_2',
 '№ Группы': 'col_3',
 '№ Объявления': 'col_4',
 'Формат': 'col_5',
 'Размер изображения': 'col_6',
 'Тип устройства': 'col_7',
 'Пол': 'col_8',
 'Категория таргетинга': 'col_9',
 'Упоминание брендов': 'col_10',
 'Уровень платежеспособности': 'col_11',
 'Возраст': 'col_12',
 'Заголовок': 'col_13',
 'Текст': 'col_14',
 'Показы': 'col_15',
 'Взвешенные показы': 'col_16',
 'Клики': 'col_17',
 'CTR (%)': 'col_18',
 'wCTR (%)': 'col_19',
 'Расход (руб.)': 'col_20',
 'Конверсия (%)/Товар добавлен в корзину': 'col_21',
 'Конверсия (%)/Лид с Centra Market//Динамика': 'col_22',
 'Конверсии/Товар добавлен в корзину': 'col_23',
 'Конверсии/Лид с Centra Market//Динамика': 'col_24'}

In [25]:
df_renamed.to_csv("../dataset/binary_encoding.csv")


### Вопросы для обсуждения:

- Какие колонки можно сделать целевыми?
- Какие гипотезы можно проверить на данных?
- Какие модели можно построить на основе предложенных данных?
